# Extracción MongoDB → Excel (columnas seleccionadas)
**Filtro:** `cola_id = 43` | Fecha: hoy automático (hora Perú UTC-5)

Columnas extraídas del schema `reportes.log_comunicaciones`.

In [46]:
# pip install pymongo pandas openpyxl dnspython
import json
from datetime import datetime, timezone, timedelta
from pathlib import Path
from zoneinfo import ZoneInfo

import pandas as pd
from pymongo import MongoClient
from bson import ObjectId

pd.set_option('display.max_columns', None)

LIMA_TZ = ZoneInfo("America/Lima")

In [47]:
# ── CONFIGURACIÓN ─────────────────────────────────────────────────────────────
host_mongo = "44.195.114.98"
user_mongo = "bi_guest"
pwd_mongo = "gu3$t202X#"
db_name_mongo = "reportes"

MONGO_URI        = f"mongodb://{user_mongo}:{pwd_mongo}@{host_mongo}:27017/{db_name_mongo}"    # <-- tu URI
DATABASE     = "reportes"
COLLECTION   = "log_comunicaciones"
COLA_ID      = 2
OUTPUT_EXCEL = Path(r"C:\Users\USUARIO\Downloads\NTP_CODEX\extraccion_campos_hoy.xlsx")

In [48]:
# ── RANGO DE FECHA FIJA (08/04/2026, hora Perú UTC-5) ────────────────────────
# 00:00:00 Perú = 05:00:00 UTC  |  23:59:59 Perú = 04:59:59 UTC día siguiente
hoy_inicio = datetime(2026, 4, 8,  5,  0,  0, tzinfo=timezone.utc)
hoy_fin    = datetime(2026, 4, 9,  4, 59, 59, tzinfo=timezone.utc)

print(f"Rango UTC  : {hoy_inicio}  →  {hoy_fin}")
print(f"Fecha Perú : 2026-04-08")

Rango UTC  : 2026-04-08 05:00:00+00:00  →  2026-04-09 04:59:59+00:00
Fecha Perú : 2026-04-08


In [49]:
# ── CONEXIÓN Y EXTRACCIÓN ─────────────────────────────────────────────────────
client = MongoClient(MONGO_URI, serverSelectionTimeoutMS=8000)

try:
    client.admin.command('ping')
    print("Conexión exitosa")
except Exception as e:
    print(f"Error: {e}")
    raise

col = client[DATABASE][COLLECTION]

filtro = {
    "cola_id": COLA_ID,
    "fecha_inicio_comunicacion": {"$gte": hoy_inicio, "$lte": hoy_fin}
}

# Proyección MongoDB: solo traer los campos necesarios
proyeccion = {
    "fecha_inicio_comunicacion" : 1,
    "identificacion"            : 1,
    "tipo_comunicacion_id"      : 1,
    "marcador_info.dato1"       : 1,
    "marcador_info.dato2"       : 1,
    "marcador_info.dato3"       : 1,
    "marcador_info.dato4"       : 1,
    "marcador_info.dato5"       : 1,
    "marcador_info.lote_nombre" : 1,
    "marcador_info.lote_id"     : 1,
    "lote_nombre"               : 1,
    "lote_id"                   : 1,
    "tiempo_timbrado_marcador"  : 1,
    "tiempo_ring"               : 1,
    "tiempo_ring_agente"        : 1,
    "tiempo_ring_acumulado"     : 1,
    "tiempo_cola"               : 1,
    "tiempo_conversacion"       : 1,
    "tiempo_gestion"            : 1,
    "tiempo_atencion"           : 1,
    "fecha_inicio_atencion"     : 1,
    "fecha_fin_atencion"        : 1,
    "resultado_id"              : 1,
    "resultado_marcador_nombre" : 1,
    "finalizada_por"            : 1,
    "agente_id"                 : 1,
    "gestiones"                 : 1,
}

docs = list(col.find(filtro, proyeccion))
print(f"Documentos encontrados: {len(docs)}")

Conexión exitosa
Documentos encontrados: 13105


In [50]:
# ── DIAGNÓSTICO: lote en todas sus ubicaciones del schema ────────────────────
print("── Diagnóstico lote_nombre en todos los campos ──\n")

con_lote_raiz        = sum(1 for d in docs if d.get("lote_nombre"))
con_lote_marcador    = sum(1 for d in docs if (d.get("marcador_info") or {}).get("lote_nombre"))
con_lote_contacto_id = sum(1 for d in docs if (d.get("contacto") or {}).get("lote_id"))
con_lote_id_raiz     = sum(1 for d in docs if d.get("lote_id"))

print(f"  lote_nombre (raíz)           : {con_lote_raiz}")
print(f"  marcador_info.lote_nombre    : {con_lote_marcador}")
print(f"  lote_id (raíz)               : {con_lote_id_raiz}")
print(f"  contacto.lote_id             : {con_lote_contacto_id}")

# Docs sin lote_nombre en raíz
sin_lote_raiz = [d for d in docs if not d.get("lote_nombre")]
print(f"\n  Docs sin lote_nombre (raíz)  : {len(sin_lote_raiz)}")

# De esos, cuántos tienen lote_id en raíz
con_lote_id_de_vacios = sum(1 for d in sin_lote_raiz if d.get("lote_id"))
print(f"  → de esos, con lote_id raíz  : {con_lote_id_de_vacios}")

# De esos, cuántos tienen contacto.lote_id
con_contacto_lote = sum(1 for d in sin_lote_raiz if (d.get("contacto") or {}).get("lote_id"))
print(f"  → de esos, con contacto.lote_id: {con_contacto_lote}")

# Muestra keys y ejemplo del primer doc sin lote
if sin_lote_raiz:
    ej = sin_lote_raiz[0]
    print(f"\nKeys del primer doc sin lote_nombre:")
    print(sorted(ej.keys()))
    print(f"\nValor lote_id raíz     : {ej.get('lote_id')}")
    print(f"Valor contacto.lote_id : {(ej.get('contacto') or {}).get('lote_id')}")
    print(f"marcador_info keys     : {list((ej.get('marcador_info') or {}).keys())}")

── Diagnóstico lote_nombre en todos los campos ──

  lote_nombre (raíz)           : 10735
  marcador_info.lote_nombre    : 16
  lote_id (raíz)               : 13077
  contacto.lote_id             : 0

  Docs sin lote_nombre (raíz)  : 2370
  → de esos, con lote_id raíz  : 2342
  → de esos, con contacto.lote_id: 0

Keys del primer doc sin lote_nombre:
['_id', 'agente_id', 'fecha_fin_atencion', 'fecha_inicio_atencion', 'fecha_inicio_comunicacion', 'finalizada_por', 'gestiones', 'identificacion', 'lote_id', 'lote_nombre', 'marcador_info', 'resultado_id', 'resultado_marcador_nombre', 'tiempo_atencion', 'tiempo_cola', 'tiempo_conversacion', 'tiempo_gestion', 'tiempo_ring', 'tiempo_ring_acumulado', 'tiempo_ring_agente', 'tiempo_timbrado_marcador', 'tipo_comunicacion_id']

Valor lote_id raíz     : 280
Valor contacto.lote_id : None
marcador_info keys     : ['dato5', 'dato4', 'dato3', 'lote_nombre', 'dato2', 'dato1', 'lote_id']


In [51]:
# ── NORMALIZACIÓN ─────────────────────────────────────────────────────────────
def safe(val):
    """Convierte tipos no serializables a string."""
    if isinstance(val, ObjectId):
        return str(val)
    if isinstance(val, datetime):
        return val.strftime('%Y-%m-%d %H:%M:%S')
    if isinstance(val, (list, dict)):
        return json.dumps(val, default=str, ensure_ascii=False)
    return val


def to_lima_date(val):
    """Retorna la FECHA (datetime.date) en hora Perú."""
    if not isinstance(val, datetime):
        return None
    if val.tzinfo is None:
        val = val.replace(tzinfo=timezone.utc)
    return val.astimezone(LIMA_TZ).date()


def to_lima_time(val):
    """Retorna la HORA como string HH:MM:SS en hora Perú."""
    if not isinstance(val, datetime):
        return None
    if val.tzinfo is None:
        val = val.replace(tzinfo=timezone.utc)
    return val.astimezone(LIMA_TZ).strftime('%H:%M:%S')


def flatten_doc(doc: dict) -> dict:
    marcador  = doc.get("marcador_info") or {}
    gestiones = doc.get("gestiones")     or []
    g         = gestiones[-1] if gestiones else {}

    hora_inicio_atencion = to_lima_time(doc.get("fecha_inicio_atencion"))

    return {
        # ── fecha/hora inicio comunicación
        "fecha_inicio_comunicacion" : to_lima_date(doc.get("fecha_inicio_comunicacion")),
        "hora_inicio_comunicacion"  : to_lima_time(doc.get("fecha_inicio_comunicacion")),
        # ── identificación
        "telefono"                  : safe(doc.get("identificacion")),
        "tipo_comunicacion_id"      : safe(doc.get("tipo_comunicacion_id")),
        "lote_id"                   : safe(doc.get("lote_id")),
        # ── tiempos
        "tiempo_timbrado_marcador"  : safe(doc.get("tiempo_timbrado_marcador")),
        "tiempo_ring"               : safe(doc.get("tiempo_ring")),
        "tiempo_ring_agente"        : safe(doc.get("tiempo_ring_agente")),
        "tiempo_ring_acumulado"     : safe(doc.get("tiempo_ring_acumulado")),
        "tiempo_cola"               : safe(doc.get("tiempo_cola")),
        "tiempo_conversacion"       : safe(doc.get("tiempo_conversacion")),
        "tiempo_gestion"            : safe(doc.get("tiempo_gestion")),
        "tiempo_atencion"           : safe(doc.get("tiempo_atencion")),
        # ── fecha/hora inicio atención
        "fecha_inicio_atencion"     : to_lima_date(doc.get("fecha_inicio_atencion")),
        "hora_inicio_atencion"      : hora_inicio_atencion,
        "hora_inicio_conversacion"  : hora_inicio_atencion,
        # ── fecha/hora fin atención
        "fecha_fin_atencion"        : to_lima_date(doc.get("fecha_fin_atencion")),
        "hora_fin_atencion"         : to_lima_time(doc.get("fecha_fin_atencion")),
        # ── resultado
        "resultado_id"              : safe(doc.get("resultado_id")),
        "resultado_marcador_nombre" : safe(doc.get("resultado_marcador_nombre")),
        "finalizada_por"            : safe(doc.get("finalizada_por")),
        "agente_id"                 : safe(doc.get("agente_id")),
        # ── marcador_info
        "base_nombre"               : safe(marcador.get("dato2")),
        "documento"                 : safe(marcador.get("dato1")),
        "departamento"              : safe(marcador.get("dato3")),
        "provincia"                 : safe(marcador.get("dato4")),
        "distrito"                  : safe(marcador.get("dato5")),
        # ── gestiones (última gestión)
        "tipificacion_nivel_1"      : safe(g.get("nivel1_nombre")),
        "tipificacion_nivel_2"      : safe(g.get("nivel2_nombre")),
        "tipificacion_nivel_3"      : safe(g.get("nivel3_nombre")),
        "tipificacion_nivel_4"      : safe(g.get("nivel4_nombre")),
    }


print("Función de normalización lista.")

Función de normalización lista.


In [52]:
# ── CONSTRUIR DATAFRAME ───────────────────────────────────────────────────────
COLUMNAS = [
    "fecha_inicio_comunicacion",
    "hora_inicio_comunicacion",
    "tramo_inicio",
    "telefono",
    "tipo_comunicacion_id",
    "base_nombre",
    "lote_id",
    "tiempo_timbrado_marcador",
    "tiempo_ring",
    "tiempo_ring_agente",
    "tiempo_ring_acumulado",
    "tiempo_cola",
    "tiempo_conversacion",
    "tiempo_gestion",
    "tiempo_atencion",
    "tiempo_espera",
    "fecha_inicio_atencion",
    "hora_inicio_atencion",
    "hora_inicio_conversacion",
    "hora_fin_conversacion",
    "fecha_fin_atencion",
    "hora_fin_atencion",
    "resultado_id",
    "resultado_marcador_nombre",
    "finalizada_por",
    "documento",
    "departamento",
    "provincia",
    "distrito",
    "agente_id",
    "tipificacion_nivel_1",
    "tipificacion_nivel_2",
    "tipificacion_nivel_3",
    "tipificacion_nivel_4",
]

if docs:
    df = pd.DataFrame([flatten_doc(d) for d in docs])
else:
    df = pd.DataFrame(columns=COLUMNAS)

# ── Convertir campos numericos ────────────────────────────────────────────────
df["tipo_comunicacion_id"]     = pd.to_numeric(df["tipo_comunicacion_id"],     errors="coerce")
df["tiempo_cola"]               = pd.to_numeric(df["tiempo_cola"],               errors="coerce")
df["tiempo_ring_acumulado"]     = pd.to_numeric(df["tiempo_ring_acumulado"],     errors="coerce")
df["tiempo_conversacion"]       = pd.to_numeric(df["tiempo_conversacion"],       errors="coerce")
df["tiempo_gestion"]            = pd.to_numeric(df["tiempo_gestion"],            errors="coerce")
df["resultado_id"]              = pd.to_numeric(df["resultado_id"],              errors="coerce")
df["tiempo_timbrado_marcador"]  = pd.to_numeric(df["tiempo_timbrado_marcador"],  errors="coerce")

# ── Funcion auxiliar: suma segundos a HH:MM:SS ────────────────────────────────
def sumar_segundos(hora_str, segundos):
    if pd.isna(hora_str) or pd.isna(segundos):
        return None
    try:
        h, m, s   = map(int, str(hora_str).split(":"))
        base      = timedelta(hours=h, minutes=m, seconds=s)
        resultado = base + timedelta(seconds=int(segundos))
        total_s   = int(resultado.total_seconds())
        return f"{total_s // 3600:02d}:{(total_s % 3600) // 60:02d}:{total_s % 60:02d}"
    except Exception:
        return None

# ── tiempo_espera ─────────────────────────────────────────────────────────────
df["tiempo_espera"] = df.apply(
    lambda r: r["tiempo_cola"]
              if r["tipo_comunicacion_id"] == 2
              else r["tiempo_cola"] - r["tiempo_ring_acumulado"],
    axis=1
)

# ── hora_fin_conversacion ─────────────────────────────────────────────────────
df["hora_fin_conversacion"] = df.apply(
    lambda r: sumar_segundos(r["hora_inicio_comunicacion"], r["tiempo_conversacion"]),
    axis=1
)

# ── hora_fin_atencion ─────────────────────────────────────────────────────────
def calc_hora_fin_atencion(r):
    if r["resultado_id"] == 3:
        segundos = (r["tiempo_conversacion"] or 0) + (r["tiempo_gestion"] or 0)
    else:
        segundos = r["tiempo_gestion"] or 0
    return sumar_segundos(r["hora_inicio_atencion"], segundos)

df["hora_fin_atencion"] = df.apply(calc_hora_fin_atencion, axis=1)

# ── Forzar int64 con fillna(0): NaN -> 0 y tipo igual a tiempo_conversacion ──
ref_dtype = df["tiempo_conversacion"].dtype
for col in ["tiempo_timbrado_marcador", "tiempo_ring_acumulado", "tiempo_espera"]:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).astype(ref_dtype)

# ── tramo_inicio: tramo de 30 min basado en hora_inicio_comunicacion ─────────
def calc_tramo(hora_str):
    if pd.isna(hora_str) or hora_str is None:
        return None
    try:
        h, m, _ = map(int, str(hora_str).split(":"))
        tramo_m = 0 if m < 30 else 30
        return f"{h:02d}:{tramo_m:02d}"
    except Exception:
        return None

df["tramo_inicio"] = df["hora_inicio_comunicacion"].apply(calc_tramo)

# Garantizar que todas las columnas existen aunque esten vacias
for c in COLUMNAS:
    if c not in df.columns:
        df[c] = None

df = df[COLUMNAS]

print(f"Shape: {df.shape}")
print(f"tiempo_timbrado_marcador dtype : {df['tiempo_timbrado_marcador'].dtype}")
print(f"tiempo_ring_acumulado dtype    : {df['tiempo_ring_acumulado'].dtype}")
print(f"tiempo_espera dtype            : {df['tiempo_espera'].dtype}")
print(f"tiempo_conversacion dtype      : {df['tiempo_conversacion'].dtype}")
df.head(5)

Shape: (13105, 34)
tiempo_timbrado_marcador dtype : int64
tiempo_ring_acumulado dtype    : int64
tiempo_espera dtype            : int64
tiempo_conversacion dtype      : int64


,fecha_inicio_comunicacion,hora_inicio_comunicacion,tramo_inicio,telefono,tipo_comunicacion_id,base_nombre,lote_id,tiempo_timbrado_marcador,tiempo_ring,tiempo_ring_agente,tiempo_ring_acumulado,tiempo_cola,tiempo_conversacion,tiempo_gestion,tiempo_atencion,tiempo_espera,fecha_inicio_atencion,hora_inicio_atencion,hora_inicio_conversacion,hora_fin_conversacion,fecha_fin_atencion,hora_fin_atencion,resultado_id,resultado_marcador_nombre,finalizada_por,documento,departamento,provincia,distrito,agente_id,tipificacion_nivel_1,tipificacion_nivel_2,tipificacion_nivel_3,tipificacion_nivel_4
0,2026-04-08,09:00:02,09:00,943598890,1,BBDD_ABRIL_LIMA1,280,3,0,0,0,0,0,0,0,0,None,None,None,09:00:02,None,None,3,CASILLA VOZ,,9382772,LIMA,LIMA,SAN JUAN DE MIRAFLORES,0,None,None,None,None
1,2026-04-08,09:00:02,09:00,955163704,1,BBDD_ABRIL_LIMA1,280,3,0,0,0,0,0,0,0,0,None,None,None,09:00:02,None,None,3,CASILLA VOZ,,44157257,LIMA,LIMA,SAN JUAN DE MIRAFLORES,0,None,None,None,None
2,2026-04-08,09:00:03,09:00,922652059,1,BBDD_ABRIL_LIMA1,280,2,0,0,0,0,0,0,0,0,None,None,None,09:00:03,None,None,3,CASILLA VOZ,,9714744,LIMA,LIMA,SAN JUAN DE MIRAFLORES,0,None,None,None,None
3,2026-04-08,09:00:03,09:00,954938588,1,BBDD_ABRIL_LIMA1,280,2,0,0,0,0,0,0,0,0,None,None,None,09:00:03,None,None,3,NUMERO INVALIDO,,8387058,LIMA,LIMA,SAN JUAN DE MIRAFLORES,0,None,None,None,None
4,2026-04-08,09:00:03,09:00,963977827,1,BBDD_ABRIL_LIMA1,280,2,0,0,0,0,0,0,0,0,None,None,None,09:00:03,None,None,3,CASILLA VOZ,,48102527,LIMA,LIMA,SAN JUAN DE MIRAFLORES,0,None,None,None,None


In [7]:
# ── TIPOS DE DATOS POR COLUMNA → equivalencia MySQL ──────────────────────────
import numpy as np

def pandas_to_mysql(col_name, dtype, sample_val):
    """Sugiere el tipo MySQL según dtype de pandas y nombre de columna."""
    col = col_name.lower()

    # Fechas y horas por nombre de columna
    if col.startswith("fecha_"):
        return "DATE"
    if col.startswith("hora_"):
        return "VARCHAR(8)"      # HH:MM:SS almacenado como string

    # Numéricos
    if dtype in ["int64", "int32", "int16", "int8"]:
        return "INT"
    if dtype in ["float64", "float32"]:
        return "FLOAT"

    # Booleanos
    if dtype == "bool":
        return "TINYINT(1)"

    # Por defecto string — ajusta largo según muestra
    if isinstance(sample_val, str) and len(sample_val) > 100:
        return "TEXT"
    return "VARCHAR(255)"

# Obtener muestra no nula por columna
muestra = {c: next((v for v in df[c] if v is not None and not (isinstance(v, float) and np.isnan(v))), None)
           for c in df.columns}

tipo_info = []
for col in df.columns:
    dtype     = str(df[col].dtype)
    mysql_t   = pandas_to_mysql(col, dtype, muestra[col])
    nulos     = df[col].isna().sum()
    nullable  = "YES" if nulos > 0 else "NO"
    tipo_info.append({
        "columna"       : col,
        "dtype_pandas"  : dtype,
        "tipo_mysql"    : mysql_t,
        "nullable"      : nullable,
        "ejemplo"       : str(muestra[col])[:50] if muestra[col] is not None else "NULL",
    })

df_tipos = pd.DataFrame(tipo_info)
print(f"Total columnas: {len(df_tipos)}\n")
display(df_tipos)

# ── DDL sugerido para MySQL ───────────────────────────────────────────────────
print("\n── CREATE TABLE sugerido ──\n")
ddl_cols = []
for _, row in df_tipos.iterrows():
    null_str = "NULL" if row["nullable"] == "YES" else "NOT NULL"
    ddl_cols.append(f"    `{row['columna']}` {row['tipo_mysql']} {null_str}")

ddl = "CREATE TABLE log_comunicaciones_43 (\n"
ddl += ",\n".join(ddl_cols)
ddl += "\n);"
print(ddl)

Total columnas: 33



,columna,dtype_pandas,tipo_mysql,nullable,ejemplo
0,fecha_inicio_comunicacion,object,DATE,NO,2026-04-08
1,hora_inicio_comunicacion,object,VARCHAR(8),NO,09:56:50
2,telefono,object,VARCHAR(255),NO,988398346
3,tipo_comunicacion_id,int64,INT,NO,1
4,base_nombre,object,VARCHAR(255),YES,BBDD_MARZO_LIMA2
5,lote_id,int64,INT,NO,281
6,tiempo_timbrado_marcador,float64,FLOAT,YES,6.0
7,tiempo_ring,int64,INT,NO,0
8,tiempo_ring_agente,int64,INT,NO,0
9,tiempo_ring_acumulado,float64,FLOAT,YES,2.0



── CREATE TABLE sugerido ──

CREATE TABLE log_comunicaciones_43 (
    `fecha_inicio_comunicacion` DATE NOT NULL,
    `hora_inicio_comunicacion` VARCHAR(8) NOT NULL,
    `telefono` VARCHAR(255) NOT NULL,
    `tipo_comunicacion_id` INT NOT NULL,
    `base_nombre` VARCHAR(255) NULL,
    `lote_id` INT NOT NULL,
    `tiempo_timbrado_marcador` FLOAT NULL,
    `tiempo_ring` INT NOT NULL,
    `tiempo_ring_agente` INT NOT NULL,
    `tiempo_ring_acumulado` FLOAT NULL,
    `tiempo_cola` INT NOT NULL,
    `tiempo_conversacion` INT NOT NULL,
    `tiempo_gestion` INT NOT NULL,
    `tiempo_atencion` INT NOT NULL,
    `tiempo_espera` FLOAT NULL,
    `fecha_inicio_atencion` DATE NULL,
    `hora_inicio_atencion` VARCHAR(8) NULL,
    `hora_inicio_conversacion` VARCHAR(8) NULL,
    `hora_fin_conversacion` VARCHAR(8) NOT NULL,
    `fecha_fin_atencion` DATE NULL,
    `hora_fin_atencion` VARCHAR(8) NULL,
    `resultado_id` INT NOT NULL,
    `resultado_marcador_nombre` VARCHAR(255) NULL,
    `finalizada_por`

In [53]:
# ── ANÁLISIS DE NULOS ─────────────────────────────────────────────────────────
total = len(df)

resumen = pd.DataFrame({
    "columna"    : df.columns,
    "con_dato"   : [df[c].notna().sum() for c in df.columns],
    "nulos"      : [df[c].isna().sum()  for c in df.columns],
})
resumen["%_nulo"]     = (resumen["nulos"]    / max(total, 1) * 100).round(1)
resumen["%_con_dato"] = (resumen["con_dato"] / max(total, 1) * 100).round(1)
resumen = resumen.sort_values("%_nulo").reset_index(drop=True)

print(f"Total registros     : {total}")
print(f"Columnas con datos  : {(resumen['con_dato'] > 0).sum()} / {len(COLUMNAS)}")
print(f"Columnas 100% vacías: {(resumen['%_nulo'] == 100).sum()}")
resumen

Total registros     : 13105
Columnas con datos  : 34 / 34
Columnas 100% vacías: 0


,columna,con_dato,nulos,%_nulo,%_con_dato
0,fecha_inicio_comunicacion,13105,0,0.0,100.0
1,hora_inicio_comunicacion,13105,0,0.0,100.0
2,tramo_inicio,13105,0,0.0,100.0
3,telefono,13105,0,0.0,100.0
4,tipo_comunicacion_id,13105,0,0.0,100.0
5,lote_id,13105,0,0.0,100.0
6,tiempo_timbrado_marcador,13105,0,0.0,100.0
7,tiempo_ring,13105,0,0.0,100.0
8,tiempo_conversacion,13105,0,0.0,100.0
9,tiempo_ring_agente,13105,0,0.0,100.0


In [54]:
# ── EXPORTAR A EXCEL ──────────────────────────────────────────────────────────
with pd.ExcelWriter(OUTPUT_EXCEL, engine="openpyxl") as writer:

    # Hoja 1: datos completos
    df.to_excel(writer, sheet_name="datos", index=False)

    # Hoja 2: resumen de nulos
    resumen.to_excel(writer, sheet_name="resumen_nulos", index=False)

print(f"Excel guardado en  : {OUTPUT_EXCEL}")
print(f"  Hoja 'datos'         : {df.shape[0]} filas x {df.shape[1]} columnas")
print(f"  Hoja 'resumen_nulos' : {len(resumen)} columnas analizadas")

Excel guardado en  : C:\Users\USUARIO\Downloads\NTP_CODEX\extraccion_campos_hoy.xlsx
  Hoja 'datos'         : 13105 filas x 34 columnas
  Hoja 'resumen_nulos' : 34 columnas analizadas


In [41]:
# ── INSERTAR DATAFRAME EN MARIADB ────────────────────────────────────────────
# pip install sqlalchemy pymysql
import urllib.parse
from sqlalchemy import create_engine, text
import numpy as np

# ── Configuracion MariaDB
DB_HOST     = "intranetpbx.net.pe"
DB_PORT     = 33306
DB_USER     = "biuser"
DB_PASSWORD = "{b1us3r;3v0x}"
DB_NAME     = "db_ntp_win_outbound"
DB_TABLE    = "fact_comunicaciones"

# Codifica la contrasena para la URL (maneja caracteres especiales)
pwd_encoded = urllib.parse.quote_plus(DB_PASSWORD)
engine = create_engine(
    f"mysql+pymysql://{DB_USER}:{pwd_encoded}@{DB_HOST}:{DB_PORT}/{DB_NAME}",
    connect_args={"charset": "utf8mb4"}
)

# Verificar conexion
try:
    with engine.connect() as conn:
        conn.execute(text("SELECT 1"))
    print("Conexion MariaDB exitosa")
except Exception as e:
    print(f"Error de conexion: {e}")
    raise

# ── Preparar DataFrame para insercion ────────────────────────────────────────
df_insert = df.copy()

# Convertir Int64 nullable a int64 estandar (pymysql no acepta pd.NA)
for col in df_insert.select_dtypes(include=["Int64", "Int32"]).columns:
    df_insert[col] = df_insert[col].astype("float64").fillna(0).astype("int64")

# Convertir datetime.date a string YYYY-MM-DD
import datetime
for col in df_insert.columns:
    if df_insert[col].apply(lambda x: isinstance(x, datetime.date)).any():
        df_insert[col] = df_insert[col].apply(
            lambda x: x.strftime("%Y-%m-%d") if isinstance(x, datetime.date) else None
        )

# Reemplazar NaN/None por None (NULL en MySQL)
df_insert = df_insert.where(pd.notnull(df_insert), None)

# ── Insertar en la tabla ──────────────────────────────────────────────────────
try:
    df_insert.to_sql(
        name      = DB_TABLE,
        con       = engine,
        if_exists = "append",   # agrega filas sin borrar la tabla
        index     = False,
        chunksize = 500,         # inserta en bloques de 500 filas
        method    = "multi",     # inserts por lote (mas rapido)
    )
    print(f"Insercion exitosa: {len(df_insert)} filas insertadas en {DB_NAME}.{DB_TABLE}")
except Exception as e:
    print(f"Error al insertar: {e}")
    raise
finally:
    engine.dispose()

Conexion MariaDB exitosa
Insercion exitosa: 7084 filas insertadas en db_ntp_win_outbound.fact_comunicaciones
